# DATA 266 Lab 2 - Part 2: LoRA fine-tuning
Run all cells on the RTX 4050. The first cell prints the commit, versions, GPU, and seed. The smoke test runs before any download or training.

In [ ]:
from pathlib import Path
import subprocess,sys,platform,yaml,os
candidates=[Path.cwd(),*Path.cwd().parents]
ROOT=next((p for p in candidates if (p/'part2_diffusion').is_dir()),Path('/content/Lab-2'))
if not (ROOT/'part2_diffusion').is_dir(): subprocess.run(['git','clone','--depth','1','https://github.com/denishatank12/Lab-2.git',str(ROOT)],check=True)
P2=ROOT/'part2_diffusion'; SEED=266
print('repo:',ROOT)
print('commit:',subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('python:',platform.python_version(),'seed:',SEED)
try:
 import torch; print('torch:',torch.__version__,'gpu:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as e: print('torch diagnostic:',e)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(P2/'requirements.txt')],check=True)
print('dependencies ready')

In [ ]:
subprocess.run([sys.executable,str(P2/'src'/'smoke_test.py')],check=True)
print('SMOKE TEST PASSED - training is allowed to start')

In [ ]:
data_dir=P2/'data'/'processed'
if not (data_dir/'train').exists(): subprocess.run([sys.executable,str(P2/'src'/'prepare_data.py'),'--output-dir',str(data_dir)],check=True)
print('prepared data:',data_dir)

In [ ]:
# Both runs use the same split and step budget. Each command resumes from latest checkpoint when one exists.
for run,rank in [('run1',4),('run2',8)]:
    out=P2/'outputs'/run; args=[sys.executable,str(P2/'src'/'train_lora.py'),'--config',str(P2/'config.yaml'),'--run',run,'--rank',str(rank)]
    if any(out.glob('checkpoint-*')): args.append('--resume-latest')
    subprocess.run(args,check=True)
print('both LoRA runs completed or resumed')

In [ ]:
subprocess.run([sys.executable,str(P2/'src'/'generate_eval.py'),'--config',str(P2/'config.yaml')],check=True)
subprocess.run([sys.executable,str(P2/'src'/'evaluate.py'),'--config',str(P2/'config.yaml')],check=True)
print('generation manifest and evaluation artifact written under part2_diffusion/outputs')